# FashionMNIST Naive ANN on Colab

Notebook này chạy baseline naive đúng nghĩa: chỉ dùng raw pixel 784 chiều, không HOG, không deslant, không recenter, không augmentation.

In [1]:
import os
from pathlib import Path

IN_COLAB = 'COLAB_GPU' in os.environ
save_root = Path('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    save_root = Path('/content/drive/MyDrive/Colab Experiments/Lab3-LT')

artifact_dir = save_root / 'artifacts' / 'fashion_mnist_ann_naive'
data_root = save_root / 'data'
artifact_dir.mkdir(parents=True, exist_ok=True)
data_root.mkdir(parents=True, exist_ok=True)

print('IN_COLAB =', IN_COLAB)
print('artifact_dir =', artifact_dir)
print('data_root =', data_root)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
IN_COLAB = True
artifact_dir = /content/drive/MyDrive/Colab Experiments/Lab3-LT/artifacts/fashion_mnist_ann_naive
data_root = /content/drive/MyDrive/Colab Experiments/Lab3-LT/data


In [2]:
!nvidia-smi || true
!python -m pip install -q torch torchvision numpy

Thu Mar 26 19:28:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import FashionMNIST

class RawPixelDataset(Dataset):
    def __init__(self, images, labels):
        self.images = images.astype(np.float32).reshape(len(labels), -1) / 255.0
        self.labels = labels.astype(np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return torch.from_numpy(self.images[idx]).float(), torch.tensor(self.labels[idx]).long()

class NaiveFashionAnn(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        return self.net(x)

train_ds = FashionMNIST(root=str(data_root), train=True, download=True)
test_ds = FashionMNIST(root=str(data_root), train=False, download=True)

train_loader = DataLoader(RawPixelDataset(train_ds.data.numpy(), train_ds.targets.numpy()), batch_size=256, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(RawPixelDataset(test_ds.data.numpy(), test_ds.targets.numpy()), batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = NaiveFashionAnn().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

history = []
for epoch in range(1, 51):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_samples += labels.size(0)
    row = {
        'epoch': epoch,
        'train_loss': total_loss / total_samples,
        'train_acc': total_correct / total_samples,
    }
    history.append(row)
    print(json.dumps(row, ensure_ascii=False))

model.eval()
confusion = np.zeros((10, 10), dtype=np.int64)
all_targets, all_preds, all_probs = [], [], []
total_correct = 0
total_samples = 0
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        logits = model(images)
        probs = torch.softmax(logits, dim=1)
        preds = logits.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)
        labels_np = labels.cpu().numpy()
        preds_np = preds.cpu().numpy()
        probs_np = probs.cpu().numpy()
        all_targets.append(labels_np)
        all_preds.append(preds_np)
        all_probs.append(probs_np)
        for t, p in zip(labels_np, preds_np):
            confusion[t, p] += 1

per_class_accuracy = {}
for i in range(10):
    total_in_class = confusion[i].sum()
    per_class_accuracy[str(i)] = float(confusion[i, i] / total_in_class) if total_in_class > 0 else 0.0

final_train_acc = history[-1]['train_acc']
test_acc = total_correct / total_samples
metrics = {
    'final_train_acc': final_train_acc,
    'final_train_loss': history[-1]['train_loss'],
    'test_acc': test_acc,
    'generalization_gap': final_train_acc - test_acc,
    'per_class_accuracy': per_class_accuracy,
    'epochs_ran': len(history),
    'history': history,
}

torch.save({'model_state_dict': {k: v.cpu() for k, v in model.state_dict().items()}, 'metrics': metrics}, artifact_dir / 'model_final.pt')
(artifact_dir / 'metrics.json').write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding='utf-8')
(artifact_dir / 'class_accuracy.json').write_text(json.dumps(per_class_accuracy, indent=2, ensure_ascii=False), encoding='utf-8')
np.save(artifact_dir / 'confusion_matrix.npy', confusion)
np.save(artifact_dir / 'test_targets.npy', np.concatenate(all_targets, axis=0))
np.save(artifact_dir / 'test_predictions.npy', np.concatenate(all_preds, axis=0))
np.save(artifact_dir / 'test_probabilities.npy', np.concatenate(all_probs, axis=0))

print(json.dumps({'final_train_acc': final_train_acc, 'test_acc': test_acc, 'generalization_gap': final_train_acc - test_acc}, ensure_ascii=False))

{"epoch": 1, "train_loss": 0.5762494110266367, "train_acc": 0.7928333333333333}
{"epoch": 2, "train_loss": 0.3777887634118398, "train_acc": 0.8625166666666667}
{"epoch": 3, "train_loss": 0.3283480796178182, "train_acc": 0.8794333333333333}
{"epoch": 4, "train_loss": 0.3039381296157837, "train_acc": 0.88705}
{"epoch": 5, "train_loss": 0.2886510638554891, "train_acc": 0.8925333333333333}
{"epoch": 6, "train_loss": 0.27122574469248456, "train_acc": 0.8986666666666666}
{"epoch": 7, "train_loss": 0.2615129466215769, "train_acc": 0.9030166666666667}
{"epoch": 8, "train_loss": 0.24492916820049285, "train_acc": 0.9085}
{"epoch": 9, "train_loss": 0.23615307269891103, "train_acc": 0.9114}
{"epoch": 10, "train_loss": 0.22252540238698323, "train_acc": 0.9160333333333334}
{"epoch": 11, "train_loss": 0.2173190987586975, "train_acc": 0.9181666666666667}
{"epoch": 12, "train_loss": 0.20858236049811046, "train_acc": 0.9214166666666667}
{"epoch": 13, "train_loss": 0.2000311292807261, "train_acc": 0.9240

In [4]:
import json
metrics = json.loads((artifact_dir / 'metrics.json').read_text(encoding='utf-8'))
print('final_train_acc =', metrics['final_train_acc'])
print('test_acc =', metrics['test_acc'])
print('generalization_gap =', metrics['generalization_gap'])
print('artifact_dir =', artifact_dir)

final_train_acc = 0.9795666666666667
test_acc = 0.8959
generalization_gap = 0.08366666666666667
artifact_dir = /content/drive/MyDrive/Colab Experiments/Lab3-LT/artifacts/fashion_mnist_ann_naive
